# Brain morphometry and age prediction

Exploratory PCA and regression using right-hemisphere features. The full dataset is stored in `../data/largedataset_clean.txt`.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split, cross_val_score, learning_curve
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

DATA_PATH = Path("data/largedataset_clean.txt")
if not DATA_PATH.exists():
    DATA_PATH = Path("../data/largedataset_clean.txt")


In [ ]:
data = pd.read_csv(DATA_PATH, sep="\t")
left_columns = [c for c in data.columns if c.lower().startswith("lh_") or "left" in c.lower()]
data = data.drop(columns=left_columns + ["subejct_label"], errors="ignore")
data = data.dropna().copy()

# Predict age from right-hemisphere morphometry; gender is retained only for stratifying analysis elsewhere.
y = data["age"].astype(float)
X = data.drop(columns=["age", "gender(1=male, 2=female)"], errors="ignore")
X = X.select_dtypes(include="number")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Features: {X.shape[1]}, training rows: {len(X_train)}, test rows: {len(X_test)}")


In [ ]:
model = Pipeline([
    ("scale", StandardScaler()),
    ("pca", PCA(n_components=20)),
    ("regression", LinearRegression()),
])
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print("MAE:", mean_absolute_error(y_test, y_pred))
print("RMSE:", mean_squared_error(y_test, y_pred) ** 0.5)
print("R²:", r2_score(y_test, y_pred))


PCA and scaling are fitted only on the training split because they are inside the pipeline. Cross-validation also refits the complete pipeline within each fold. The held-out test score remains an estimate for this split; it does not establish clinical validity.


In [ ]:
cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring="r2")
print("Training-fold CV R²:", cv_scores)
print("Mean CV R²:", cv_scores.mean())


In [ ]:
train_sizes, train_scores, valid_scores = learning_curve(
    model, X_train, y_train, cv=5, scoring="r2", n_jobs=-1
)
plt.plot(train_sizes, train_scores.mean(axis=1), label="Training R²")
plt.plot(train_sizes, valid_scores.mean(axis=1), label="CV R²")
plt.xlabel("Training examples")
plt.ylabel("R²")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
ridge_model = Pipeline([
    ("scale", StandardScaler()),
    ("pca", PCA(n_components=20)),
    ("regression", Ridge(alpha=0.1)),
])
ridge_model.fit(X_train, y_train)
ridge_pred = ridge_model.predict(X_test)
print("Ridge test R²:", r2_score(y_test, ridge_pred))
